# Python for ML4EO · Tutorial T3
# Reading satellite images with rasterio

**Self-study · about 60 minutes · after Tutorials T0–T2**

The course images are **GeoTIFF** files: a grid of pixel values (an array, T2) plus information about **where on Earth** the grid lies.
The library **rasterio** reads both. In T2 you already used the ready-made function `read_band()`; this tutorial explains how it works. At the end of this tutorial you can read, line by line, the functions `load_band()` and `read_scl()` used in every practical.

| Part | Topic |
|---|---|
| 1 | Is rasterio installed? |
| 2 | Finding the image files |
| 3 | Opening a file and reading its metadata |
| 4 | Reading the pixel values |
| 5 | No-data values and the function `load_band()` |
| 6 | From map coordinates to pixels |
| 7 | Several bands and several dates |
| 8 | The Scene Classification map and `read_scl()` |
| 9 | Common errors |

**Data:** the Sentinel-2 files in `ml4eo/data` (dates 2026-01-21, 2026-05-13, 2026-09-03).
Each band is 1,702 × 2,380 pixels; reading one takes about 16 MB of memory. Close other programs if your computer has little memory.

Tasks: **▶ RUN** · **✎ CHANGE** · **✍ EXERCISE** (followed by a check).

### Set-up  ▶ RUN

In [ ]:
# Run this cell once. It creates the function check(), which tells you whether an exercise is correct.
def check(correct, hint):
    if correct:
        print("✓ Correct, well done!")
    else:
        print("✗ Not yet. Hint:", hint)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

---
## Part 1 · Is rasterio installed?

### 1.1 Import  ▶ RUN

In [ ]:
import rasterio
rasterio.__version__     # (output depends on your computer)

You should see a version number such as `1.4.3`.

If you get `ModuleNotFoundError: No module named 'rasterio'`:
1. Check that Jupyter was started from the `ml4eo` environment (T0, Part 1). This is the usual cause.
2. If it really is missing: close Jupyter, open the **Anaconda Prompt** and type

```
conda activate ml4eo
conda install -c conda-forge rasterio
```

On Windows, install rasterio with `conda` from `conda-forge`, not with `pip`: the conda package includes the GDAL library that rasterio needs.

---
## Part 2 · Finding the image files

### 2.1 The data folder  ▶ RUN
As in T0 and T2.

In [ ]:
DATA = Path("../data")
DATA.exists()

### 2.2 Listing the raw bands of one date  ▶ RUN
`DATA.glob(pattern)` finds all files whose name matches a **pattern**; `*` stands for "any text". `sorted()` puts them in alphabetical order.

In [ ]:
files = sorted(DATA.glob("2026-09-03*_(Raw).tiff"))
print(len(files), "files")
for f in files:
    print(f.name)

12 bands: B01–B12 and B8A (there is no B10 in Level-2A products).

### 2.3 One file name  ▶ RUN
The same f-string as in T1, 3.6. `DATA / name` joins folder and file name (T0, Part 5).

In [ ]:
date, band = "2026-09-03", "B04"
path = DATA / f"{date}-00_00_{date}-23_59_Sentinel-2_L2A_{band}_(Raw).tiff"
path.exists()

---
## Part 3 · Opening a file and reading its metadata

### 3.1 Open  ▶ RUN
`rasterio.open(path)` opens the file and returns a **dataset** object, here called `src` (for "source"). Nothing is read into memory yet.

In [ ]:
src = rasterio.open(path)
print("opened:", src.name)     # (output depends on your computer)

### 3.2 Size and number of bands  ▶ RUN

In [ ]:
print("columns (width): ", src.width)
print("rows (height):   ", src.height)
print("number of layers:", src.count)
print("data type:       ", src.dtypes)

One layer of decimal numbers (`float32`): the surface reflectance of band B04.

### 3.3 Coordinate reference system (CRS)  ▶ RUN

In [ ]:
print(src.crs)

**EPSG:32736** is the code of **WGS 84 / UTM zone 36S**: a projected coordinate system in **metres**, used for Rwanda (Lectures 1 and 3). All course layers use this CRS.

### 3.4 Pixel size  ▶ RUN
`src.res` gives the pixel size (x, y) in the units of the CRS, i.e. metres.

In [ ]:
src.res

About 19.98 m × 19.96 m: the 20 m bands, slightly resampled when the image was exported from the Copernicus Browser. In calculations we use the exact values, not "20".

### 3.5 Extent  ▶ RUN
`src.bounds` gives the coordinates of the edges: left, bottom, right, top (in metres).

In [ ]:
src.bounds

### 3.6 Size of the image on the ground  ▶ RUN
Width = right − left; height = top − bottom. Divide by 1,000 for kilometres.

In [ ]:
b = src.bounds
width_km = (b.right - b.left) / 1000
height_km = (b.top - b.bottom) / 1000
print(f"{width_km:.1f} km × {height_km:.1f} km")

### 3.7 The transform  ▶ RUN
The **affine transform** links pixel positions to map coordinates. Of its six numbers, three matter here:
the pixel width (first number), the x coordinate of the **upper-left corner** (third), the pixel height, negative because rows go **down** while y goes **up** (fifth), and the y coordinate of the upper-left corner (sixth).

In [ ]:
src.transform

### 3.8 No-data value  ▶ RUN
`src.nodata` would give the value that marks pixels without data, if the file declared one.

In [ ]:
print(src.nodata)

`None`: these files do **not** declare a no-data value. By convention, pixels without data have the value **0** (T2, Part 5). We must handle them ourselves (Part 5).

### 3.9 Close the file  ▶ RUN

In [ ]:
src.close()

### 3.10 Metadata  ✍ EXERCISE
Open the **B08** file of **2026-01-21** and store its CRS code (`src.crs.to_epsg()`, a number) in `epsg` and its number of columns in `n_cols`. Then close it.

In [ ]:
path_b08 = DATA / "2026-01-21-00_00_2026-01-21-23_59_Sentinel-2_L2A_B08_(Raw).tiff"
src = rasterio.open(path_b08)
epsg = src.crs.to_epsg()       # ✍ replace None
n_cols = src.width     # ✍ replace None
src.close()
check(epsg == 32736 and n_cols == 2380, "src.crs.to_epsg() and src.width")


---
## Part 4 · Reading the pixel values

### 4.1 The `with` block  ▶ RUN
Forgetting `src.close()` keeps the file locked (on Windows you then cannot move or delete it). A **`with` block** closes the file automatically when the indented lines are finished.
`src.read(1)` reads **layer 1** (counting starts at 1 here, not 0!) as a 2-D NumPy array.

In [ ]:
with rasterio.open(path) as src:
    red = src.read(1)
print(red.shape, red.dtype)

After the `with` block the file is closed, but the array `red` stays in memory. Everything from T2 now works on it.

### 4.2 It is a NumPy array  ▶ RUN

In [ ]:
print("pixel (800, 1200):", red[800, 1200])
print("maximum:", red.max())

### 4.3 A first look  ▶ RUN

In [ ]:
plt.imshow(red, cmap="gray", vmin=0, vmax=0.15)
plt.colorbar(label="B04 reflectance")
plt.show()

Water is dark (low red reflectance); the city centre and dry soils are bright. The black strips along some edges are pixels with value 0: no data.

### 4.4 `read()` without a number  ▶ RUN
`src.read()` reads **all** layers into a 3-D array (layers, rows, columns). For a one-band file the first dimension is 1.

In [ ]:
with rasterio.open(path) as src:
    everything = src.read()
everything.shape

---
## Part 5 · No-data values and the function `load_band()`

### 5.1 How many pixels have no data?  ▶ RUN

In [ ]:
n_zero = (red == 0).sum()
print(n_zero, "pixels with value 0 =", f"{n_zero / red.size:.2%}", "of the image")

### 5.2 Replace 0 by NaN  ▶ RUN
Exactly as in T2: the array is `float32`, so it can hold NaN.

In [ ]:
red[red == 0] = np.nan
np.nanmean(red)

### 5.3 Putting it together: `load_band()`  ▶ RUN
Steps 2.3, 4.1 and 5.2 are needed for every band of every date. So the practicals put them in a **function** (T1, Part 8). Read each comment, then run the cell.

In [ ]:
def load_band(date, band):
    # Open one Sentinel-2 band of one date: reflectance 0-1, NaN where there is no data (value 0)
    path = DATA / f"{date}-00_00_{date}-23_59_Sentinel-2_L2A_{band}_(Raw).tiff"   # the file name
    with rasterio.open(path) as src:     # open; closed automatically afterwards
        a = src.read(1)                  # layer 1 as a 2-D array
    a[a == 0] = np.nan                   # no data -> NaN
    return a                             # give the array back

(The practicals write `f"{DATA}/{date}…"` instead of `DATA / f"{date}…"`; both give the same path.)

### 5.4 Using it  ▶ RUN
Two lines instead of eight.

In [ ]:
red = load_band("2026-09-03", "B04")
nir = load_band("2026-09-03", "B08")
print(red.shape, nir.shape)

### 5.5 NDVI of the whole image  ▶ RUN

In [ ]:
ndvi = (nir - red) / (nir + red)
plt.imshow(ndvi, cmap="RdYlGn", vmin=-0.5, vmax=0.9)
plt.colorbar(label="NDVI, 3 September 2026")
plt.show()

### 5.6 Area in hectares, with the exact pixel size  ▶ RUN
`src.res` gives the pixel size; the area of a pixel is width × height.

In [ ]:
with rasterio.open(path) as src:
    px_w, px_h = src.res
pixel_area_ha = px_w * px_h / 10_000
print(f"one pixel = {pixel_area_ha:.4f} ha")

### 5.7 Dense vegetation in hectares  ✍ EXERCISE
Compute the area (ha) of all pixels with NDVI > 0.5 on 3 September 2026 and store it in `dense_ha`.

In [ ]:
dense_ha = (ndvi > 0.5).sum() * pixel_area_ha     # ✍ replace None
print(f"{dense_ha:,.0f} ha" if dense_ha is not None else "")

check(dense_ha is not None and abs(dense_ha - (ndvi > 0.5).sum() * pixel_area_ha) < 1e-6, "number of pixels × area of one pixel: (ndvi > 0.5).sum() * pixel_area_ha")


`:,.0f` adds thousands separators and shows no decimals.

---
## Part 6 · From map coordinates to pixels

### 6.1 Pixel → coordinates: `src.xy(row, col)`  ▶ RUN
Gives the map coordinates (x, y) of the **centre** of a pixel.

In [ ]:
src = rasterio.open(path)
src.xy(0, 0)          # centre of the upper-left pixel

Compare with `src.bounds` (3.5): half a pixel (about 10 m) right of `left` and below `top`.

### 6.2 Coordinates → pixel: `src.index(x, y)`  ▶ RUN
The other direction: in which (row, column) lies a point with map coordinates (x, y)?
Reference point **P001** (water) of `kigali_reference_points.csv` lies at x = 186,646 m, y = 9,779,579 m in UTM 36S.

In [ ]:
row, col = src.index(186646, 9779579)
print("row", row, "column", col)

### 6.3 The NDVI at that point  ▶ RUN

In [ ]:
print("NDVI at P001:", ndvi[row, col])

Negative: water, as recorded in the field. This is how the training table of the course was built: for every reference point (and every pixel inside a reference polygon) the band values were read at its row and column.

Note: GPS points are recorded in latitude/longitude (EPSG:4326). They must first be converted to UTM 36S, e.g. with `rasterio.warp.transform` or in QGIS. The coordinates above are already converted.

### 6.4 Your point  ✍ EXERCISE
Reference point **P003** (dense vegetation) lies at x = 177,096 m, y = 9,771,503 m. Find its row and column and store its NDVI in `ndvi_p003`.

In [ ]:
row3, col3 = src.index(177096, 9771503)     # ✍ replace None, None
ndvi_p003 = ndvi[row3, col3]            # ✍ replace None
print(f"P003 at row {row3}, col {col3}: NDVI = {ndvi_p003:.3f}")

check(ndvi_p003 is not None and abs(ndvi_p003 - ndvi[1280, 697]) < 1e-6, "row3, col3 = src.index(177096, 9771503), then ndvi[row3, col3]")


In [ ]:
src.close()

---
## Part 7 · Several bands and several dates

### 7.1 Several bands into a dictionary  ▶ RUN
A loop (T1, Part 7) and a dictionary (T1, Part 5): band name → array.

In [ ]:
bands = {}
for b in ["B02", "B03", "B04", "B08", "B11"]:
    bands[b] = load_band("2026-09-03", b)
list(bands.keys())

### 7.2 Using the dictionary  ▶ RUN
`bands["B03"]` is the green band, and so on.

In [ ]:
mndwi = (bands["B03"] - bands["B11"]) / (bands["B03"] + bands["B11"])
print(f"water (MNDWI > 0): {(mndwi > 0).sum() * pixel_area_ha:,.0f} ha")

### 7.3 True colour  ▶ RUN
Red, green, blue as the last dimension (T2, 7.4).

In [ ]:
rgb = np.dstack([bands["B04"], bands["B03"], bands["B02"]])
plt.figure(figsize=(9, 6))
plt.imshow(np.clip(rgb * 3.5, 0, 1))
plt.show()

### 7.4 Several dates  ▶ RUN
The same NDVI for the three dates. Rwanda has a long dry season (June–September) and two rainy seasons: compare January, May and September.

In [ ]:
for date in ["2026-01-21", "2026-05-13", "2026-09-03"]:
    r, n = load_band(date, "B04"), load_band(date, "B08")
    nd = (n - r) / (n + r)
    print(date, "mean NDVI:", round(float(np.nanmean(nd)), 3))

Careful: clouds also lower the NDVI, so a cloudy date has a lower mean NDVI even if the vegetation is the same. Part 8 shows how to find the clouds.

### 7.5 A change map  ✍ EXERCISE
Compute the NDVI of **13 May 2026** (`ndvi_may`) and the change map `change = ndvi - ndvi_may` (September minus May). Show it with the colour map `"RdBu"` between −0.5 and 0.5.

In [ ]:
red_may = load_band("2026-05-13", "B04")
nir_may = load_band("2026-05-13", "B08")
ndvi_may = (nir_may - red_may) / (nir_may + red_may)     # ✍ replace None
change = ndvi - ndvi_may       # ✍ replace None
# ✍ show the map: plt.imshow(...), plt.colorbar(...), plt.show()
plt.figure(figsize=(7, 5))
plt.imshow(change, cmap="RdBu", vmin=-0.5, vmax=0.5)
plt.colorbar(label="NDVI change (Sept - May)")
plt.title("NDVI Change: September vs May 2026")
plt.show()


In [ ]:
check(change is not None and np.allclose(change, ndvi - (nir_may - red_may) / (nir_may + red_may), equal_nan=True), "ndvi_may = (nir_may - red_may) / (nir_may + red_may); change = ndvi - ndvi_may")

* **Red** (most of the image): NDVI decreased from May (end of the long rainy season) to September (end of the dry season): crops harvested, grass dried.
* **White** (centre-west): little change, mainly the built-up area of Kigali, with little vegetation on either date.
* **Blue**: NDVI increased, along the wetland valley floors (possibly flooded or recently planted in May) and on parts of the lakes. On water, NDVI changes with sediment and algae, not with vegetation.

A change map is only meaningful for pixels that are clear (cloud-free) on both dates. Both dates are almost cloud-free here; Part 8 shows how to check.

---
## Part 8 · The Scene Classification map and `read_scl()`

### 8.1 A coloured picture, not numbers  ▶ RUN
The Sentinel-2 **Scene Classification (SCL)** assigns each pixel a class code: 4 vegetation, 6 water, 8–10 cloud, and so on. The course file, however, stores the SCL as a **coloured picture** with 3 layers (red, green, blue), not as codes.

In [ ]:
scl_path = DATA / "2026-09-03-00_00_2026-09-03-23_59_Sentinel-2_L2A_Scene_classification_map_.tiff"
with rasterio.open(scl_path) as src:
    print("layers:", src.count)
    scl_rgb = src.read()
print(scl_rgb.shape)

### 8.2 One pixel  ▶ RUN
The three layers hold colour intensities between 0 and 1. Multiplied by 255 they are the usual RGB colour values.

In [ ]:
np.round(scl_rgb[:, 1280, 697] * 255)     # the pixel of reference point P003 (6.4)

(0, 160, 0) is the green that the SCL legend uses for **vegetation** (code 4).

### 8.3 The colour table  ▶ RUN
A dictionary from code to colour, exactly as in the practicals.

In [ ]:
SCL_COLOURS = {0: (0, 0, 0), 1: (255, 0, 0), 2: (47, 47, 47), 3: (100, 50, 0), 4: (0, 160, 0), 5: (255, 230, 90),
               6: (0, 0, 255), 7: (128, 128, 128), 8: (192, 192, 192), 9: (255, 255, 255), 10: (100, 200, 255), 11: (255, 150, 255)}
SCL_NAMES = {0: "no data", 1: "saturated/defective", 2: "dark area", 3: "cloud shadow", 4: "vegetation", 5: "not vegetated",
             6: "water", 7: "unclassified", 8: "cloud (medium prob.)", 9: "cloud (high prob.)", 10: "thin cirrus", 11: "snow"}

### 8.4 `read_scl()`: colours back to codes  ▶ RUN
For each code in the table, the pixels whose three colour values equal that code's colour receive the code. Read the comments, then run.

In [ ]:
def read_scl(date):
    # The scene classification is stored as a coloured picture: turn each colour back into its class code
    path = DATA / f"{date}-00_00_{date}-23_59_Sentinel-2_L2A_Scene_classification_map_.tiff"
    with rasterio.open(path) as src:
        rgb = np.round(src.read() * 255)                    # 3 layers, values 0-255
    codes = np.zeros(rgb.shape[1:], dtype=np.uint8)         # an empty map (all 0) of rows × columns
    for code_, (r, g, b) in SCL_COLOURS.items():            # for every code and its colour
        codes[(rgb[0] == r) & (rgb[1] == g) & (rgb[2] == b)] = code_   # pixels of that colour get the code
    return codes

### 8.5 Using it  ▶ RUN
`np.unique(..., return_counts=True)` lists the codes that occur and how many pixels each has.

In [ ]:
scl = read_scl("2026-09-03")
codes, counts = np.unique(scl, return_counts=True)
for c, n in zip(codes, counts):
    print(f"{c:2d} {SCL_NAMES[c]:22s} {n / scl.size:6.1%}")

`zip` walks through two lists side by side. `:22s` pads the name to 22 characters, so that the columns line up.

### 8.6 A cloud mask  ▶ RUN
`np.isin(array, list)` is `True` where the value is one of the list. Clouds and shadows: codes 3, 8, 9, 10.
The image of 21 January (short rainy season) has the most clouds.

In [ ]:
scl_jan = read_scl("2026-01-21")
cloud_jan = np.isin(scl_jan, [3, 8, 9, 10])
print(f"cloud or shadow on 21 January: {cloud_jan.mean():.1%}")

### 8.7 Masking the clouds  ▶ RUN
Compute the NDVI of January and put NaN where the SCL shows cloud or shadow. The masked pixels appear white.

In [ ]:
r, n = load_band("2026-01-21", "B04"), load_band("2026-01-21", "B08")
ndvi_jan = np.where(cloud_jan, np.nan, (n - r) / (n + r))
plt.imshow(ndvi_jan, cmap="RdYlGn", vmin=-0.5, vmax=0.9)
plt.colorbar(label="NDVI, 21 January 2026, clouds masked")
plt.show()

### 8.8 Other files in the data folder
The files `…_NDVI.tiff`, `…_NDWI.tiff` and `…_True_color.tiff` are also **coloured pictures** (3 layers) for viewing, not values. Always compute indices from the `(Raw)` bands, as in this tutorial.

---
## Part 9 · Common errors

### 9.1 A file that does not exist  ▶ RUN
The next cell makes an error **on purpose**: band "B10" does not exist in Level-2A products.

In [ ]:
# This cell makes an error on purpose
x = load_band("2026-09-03", "B10")

`RasterioIOError: … No such file or directory`: the path is wrong or the file does not exist. Check the date, the band and `DATA` with the T0 checklist (`path.exists()`).

| Error | Typical cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'rasterio'` | wrong environment; rasterio not installed | start Jupyter from `ml4eo`; `conda install -c conda-forge rasterio` |
| `RasterioIOError: … No such file or directory` | wrong folder, date or band; hidden `.tiff` extension | `path.exists()`; T0 Part 6 |
| `RasterioIOError: … not recognized as a supported file format` | file incomplete (download interrupted, OneDrive "online only") | download again; "Always keep on this device" |
| `IndexError: band index 0 out of range` after `src.read(0)` | layers are counted from **1** in rasterio | `src.read(1)` |
| `PermissionError` when moving or deleting a `.tiff` | the file is still open (no `with`, no `close()`) | use `with`; restart the kernel |
| result `nan` | no-data pixels | `np.nanmean`, … |
| `MemoryError` | too many full-size bands in memory | load only the bands you need; restart the kernel |

---
## Final challenge  ✍ EXERCISE

For each of the three dates, print the share of the image covered by **cloud or shadow** (SCL codes 3, 8, 9, 10) and the **mean NDVI of the clear pixels only**.
Store the three shares in the list `cloud_shares` (in date order).

Hints: a loop over the dates (7.4); inside: `read_scl`, `np.isin`, `load_band` twice, the NDVI, then `np.nanmean(nd[~cloud])`.

In [ ]:
cloud_shares = []
for date in ["2026-01-21", "2026-05-13", "2026-09-03"]:
    scl_d = read_scl(date)
    cloud = np.isin(scl_d, [3, 8, 9, 10])     # ✍ the cloud/shadow mask of this date
    cloud_shares.append(cloud.mean())
    red_d = load_band(date, "B04")
    nir_d = load_band(date, "B08")
    nd_d = (nir_d - red_d) / (nir_d + red_d)
    mean_clear_ndvi = np.nanmean(nd_d[~cloud])
    print(f"{date}: cloud/shadow = {cloud.mean():.2%}, clear mean NDVI = {mean_clear_ndvi:.3f}")


check(len(cloud_shares) == 3 and abs(cloud_shares[1] - np.isin(read_scl("2026-05-13"), [3, 8, 9, 10]).mean()) < 1e-9, "append cloud.mean() for each date, in the order January, May, September")


Which date is best suited to mapping land cover? Does the ranking of mean NDVI change once clouds are removed (compare with 7.4)?

---
## Summary

| Task | Code |
|---|---|
| find files | `sorted(DATA.glob("2026-09-03*_(Raw).tiff"))` |
| open and close automatically | `with rasterio.open(path) as src:` |
| metadata | `src.width`, `src.height`, `src.count`, `src.crs`, `src.res`, `src.bounds`, `src.transform`, `src.nodata` |
| read one layer / all layers | `src.read(1)` (2-D) / `src.read()` (3-D) — layers counted from 1 |
| no data | `a[a == 0] = np.nan` |
| course helper | `load_band(date, band)` |
| pixel ↔ map coordinates | `src.xy(row, col)`, `src.index(x, y)` |
| pixel area | `src.res[0] * src.res[1]` m² |
| scene classification | `read_scl(date)`; `np.unique(..., return_counts=True)`; `np.isin(scl, [3, 8, 9, 10])` |

**Next:** Tutorial **T4 · pandas: tables**: the training table, the cell table and the reference points as tables; selecting columns and rows, grouping and summarising.

### 💬 Answers to Discussion Questions

**1. Which date is best suited to mapping land cover?**
- **3 September 2026** is the best suited date for mapping baseline land cover because it has the least cloud and cloud shadow contamination (only **0.90%** cloud/shadow, compared to **6.81%** in January and **1.10%** in May), providing almost 99% clear-sky valid pixels across Kigali.

**2. Does the ranking of mean NDVI change once clouds are removed (compare with 7.4)?**
- In 7.4 (without masking), January's apparent mean NDVI was artificially pulled down by bright, cold cloud pixels. Once clouds and shadows are masked out, the ranking of clear mean NDVI is:
  1. **13 May 2026:** ~0.67 (peak wet season greenness)
  2. **21 January 2026:** ~0.61 (wet season A)
  3. **3 September 2026:** ~0.32 (end of dry season)
  The true biological ranking shows that wet season months have approximately double the greenness of the dry season once cloud artifacts are removed.